In [1]:
# Setup imports & load dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os
df = pd.read_csv('data/raw_transactions.csv', na_values=['Nan', ''])


# Section 13: 50 Senior Pandas Interview Questions & Interactive Solutions
This section features 50 real-world, high-level interview questions executed against your dataset columns.

### Q1: How do you perform a deep memory profile check on our transactions dataset, showing memory in MB for string columns?

In [2]:
# Solution:
mem_bytes = df.memory_usage(deep=True)
for col in df.columns:
    print(f"Col '{col}': {mem_bytes[col] / 1e6:.4f} MB")

Col 'transaction_id': 0.8550 MB
Col 'customer_id': 0.8250 MB
Col 'merchant_id': 0.8100 MB
Col 'transaction_amount': 0.1200 MB
Col 'card_type': 0.8325 MB
Col 'transaction_status': 0.8478 MB
Col 'device_type': 0.8060 MB
Col 'account_age_months': 0.1200 MB
Col 'transaction_date': 0.9156 MB
Col 'region': 0.8031 MB
Col 'is_fraud': 0.1200 MB


### Q2: How do you downcast the numeric type of account_age_months to save memory using pd.to_numeric()?

In [3]:
# Solution:
before_type = df['account_age_months'].dtype
df['account_age_months'] = pd.to_numeric(df['account_age_months'], downcast='unsigned')
after_type = df['account_age_months'].dtype
print(f"Type changed from {before_type} to {after_type}")

Type changed from int64 to uint8


### Q3: How do you convert the geographical 'region' column into a categorical type and check the compression ratio?

In [4]:
before_size = df['region'].memory_usage(deep=True)
# Solution:
df_optimized = df.copy()
df_optimized['region'] = df_optimized['region'].astype('category')
after_size = df_optimized['region'].memory_usage(deep=True)
print(f"Before: {before_size} bytes, After: {after_size} bytes")
print(f"Savings: {(1 - after_size/before_size)*100:.2f}%")

Before: 803228 bytes, After: 16082 bytes
Savings: 98.00%


### Q4: How do you stream/process chunks of data/raw_transactions.csv to compute the sum of transaction_amount without loading it fully into memory?

In [5]:
# Solution:
total_sum = 0
for chunk in pd.read_csv('data/raw_transactions.csv', chunksize=2000):
    total_sum += pd.to_numeric(chunk['transaction_amount'], errors='coerce').sum()
print("Total sum of transaction_amount via chunks:", total_sum)

Total sum of transaction_amount via chunks: 14349538.14


### Q5: Z-Score normalization: How do you subtract the group region mean and divide by group region std within groupby without merging?

In [6]:
# Solution:
grp_mean = df.groupby('region')['transaction_amount'].transform('mean')
grp_std = df.groupby('region')['transaction_amount'].transform('std')
df['amount_z_score'] = (df['transaction_amount'] - grp_mean) / grp_std
print(df[['region', 'transaction_amount', 'amount_z_score']].head(5))

  region  transaction_amount  amount_z_score
0  North             1216.33        0.342910
1   East              324.99       -1.162672
2   East              136.66       -1.489005
3   West              124.21       -1.526963
4   East             1284.68        0.500249


### Q6: How do you filter out entire regions that have an aggregate transaction volume below a threshold (e.g. 100,000)?

In [7]:
# Solution:
result = df.groupby('region').filter(lambda g: g['transaction_amount'].sum() >= 100000)
print("Remaining regions:", result['region'].unique())

Remaining regions: ['North' 'East' 'West' 'South' 'east']


### Q7: How do you retrieve the top 3 highest-value transactions for each region using a custom apply?

In [8]:
# Solution:
result = df.groupby('region', group_keys=False).apply(lambda g: g.nlargest(3, 'transaction_amount'))
print(result[['region', 'transaction_id', 'transaction_amount']])

<string>:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
        region transaction_id  transaction_amount
8848     East        TX112796             1913.72
2637     East        TX110878             1882.23
6550     East        TX109879             1835.03
7857    North        TX113950             1982.93
7234    North        TX101294             1969.61
13451   North        TX108998             1938.18
9368    South        TX107271             1980.44
1240    South        TX103625             1939.96
11562   South        TX110922             1937.53
12417    West        TX113656             1994.40
6002     West        TX114302             1987.40
5541     West        TX101470             1962.62

### Q8: How do you compute a 7-day rolling average of transaction amounts after setting transaction_date as index?

In [9]:
df_roll = df.copy()
df_roll['transaction_date'] = pd.to_datetime(df_roll['transaction_date'], format='mixed')
df_roll = df_roll.set_index('transaction_date').sort_index()
# Solution:
rolling_avgs = df_roll['transaction_amount'].rolling('7D').mean()
print(rolling_avgs.head(15))

transaction_date
2025-01-01     240.570000
2025-01-01     899.185000
2025-01-01     999.540000
2025-01-01    1214.387500
2025-01-01    1010.804000
2025-01-01     852.288333
2025-01-01     903.365714
2025-01-01     935.263750
2025-01-01     849.214444
2025-01-01     963.574000
2025-01-01     974.345455
2025-01-01    1040.944167
2025-01-01     989.083077
2025-01-01    1043.842143
2025-01-01    1015.978000
Name: transaction_amount, dtype: float64


### Q9: How do you compute the expanding standard deviation of transaction amounts chronologically?

In [10]:
df_exp = df.copy()
df_exp['transaction_date'] = pd.to_datetime(df_exp['transaction_date'], format='mixed')
df_exp = df_exp.sort_values('transaction_date')
# Solution:
expanding_std = df_exp['transaction_amount'].expanding().std()
print(expanding_std.head(10))

8313            NaN
8702     931.422265
7175     681.165983
11388    702.824675
7038     760.067826
3600     782.895766
854      727.347080
711      679.409596
10943    685.957776
2367     740.970406
Name: transaction_amount, dtype: float64


### Q10: How do you smooth out transaction trends using Exponentially Weighted Moving Average (EWMA) with span=30?

In [11]:
df_ewm = df.copy()
df_ewm['transaction_date'] = pd.to_datetime(df_ewm['transaction_date'], format='mixed')
df_ewm = df_ewm.sort_values('transaction_date')
# Solution:
ewma_vals = df_ewm['transaction_amount'].ewm(span=30).mean()
print(ewma_vals.head(10))

8313      240.570000
8702      921.138833
7175     1020.444954
11388    1251.478139
7038     1011.438245
3600      825.247606
854       891.763636
711       933.391892
10943     822.950499
2367      978.022000
Name: transaction_amount, dtype: float64


### Q11: How do you calculate the month-over-month percentage change in total transaction amounts?

In [12]:
df_time = df.copy()
df_time['transaction_date'] = pd.to_datetime(df_time['transaction_date'], format='mixed')
df_time = df_time.set_index('transaction_date').sort_index()
# Solution:
monthly_totals = df_time['transaction_amount'].resample('ME').sum()
pct_changes = monthly_totals.pct_change() * 100
print(pct_changes.head(10))

transaction_date
2025-01-31         NaN
2025-02-28   -6.295761
2025-03-31    9.109885
2025-04-30   -1.880449
2025-05-31    5.657339
2025-06-30   -3.273906
2025-07-31    0.359979
2025-08-31    3.330496
2025-09-30   -5.661242
2025-10-31    0.914998
Freq: ME, Name: transaction_amount, dtype: float64


### Q12: How do you shift the transaction_amount column by 1 entry to find the previous transaction's amount for each customer?

In [13]:
# Solution:
df_sorted = df.sort_values(['customer_id', 'transaction_date'])
df_sorted['prev_tx_amount'] = df_sorted.groupby('customer_id')['transaction_amount'].shift(1)
print(df_sorted[['customer_id', 'transaction_date', 'transaction_amount', 'prev_tx_amount']].dropna().head(5))

      customer_id     transaction_date  transaction_amount  prev_tx_amount
12483      C10053           04/05/2026             1457.15          188.24
5837       C10053          13-Feb-2025              323.03         1975.52
3064       C10053           16/06/2025              751.65          323.03
5644       C10053  2025-08-22 03:11:28              528.81          751.65
1525       C10053           2026-03-07             1484.10          528.81


### Q13: How do you slice a MultiIndex DataFrame (indexed by region and customer_id) to select specific customers in a region?

In [14]:
df_mi = df.set_index(['region', 'customer_id']).sort_index()
# Solution:
# Extract transactions in East region for customer pool
print(df_mi.loc[pd.IndexSlice['East', :], :].head(3))

                   transaction_id merchant_id  ...  is_fraud amount_z_score
region customer_id                             ...                         
East   C10053            TX102491       M9410  ...         0      -0.809499
       C10053            TX113741       M3112  ...         0      -1.166069
       C10053            TX104680       M9176  ...         0       0.799100

[3 rows x 10 columns]


### Q14: How do you retrieve cross-sections (.xs) of transactions for a specific customer across all regions?

In [15]:
df_mi = df.set_index(['region', 'customer_id']).sort_index()
# Solution:
cust_sample = df['customer_id'].iloc[0]
print(df_mi.xs(key=cust_sample, level='customer_id').head(3))

        transaction_id merchant_id  ...  is_fraud amount_z_score
region                              ...                         
 East         TX107381       M8030  ...         0      -1.510748
 South        TX100645       M9444  ...         0      -0.754073
East          TX100582       M2215  ...         0      -0.172966

[3 rows x 10 columns]


### Q15: How do you pivot the transaction summary to show total revenue with region as index and transaction_size (Large/Small) as columns?

In [16]:
df_size = df.copy()
df_size['transaction_amount'] = pd.to_numeric(df_size['transaction_amount'], errors='coerce')
df_size['transaction_size'] = np.where(df_size['transaction_amount'] > 500, 'Large', 'Small')
# Solution:
pivoted = df_size.pivot_table(index='region', columns='transaction_size', values='transaction_amount', aggfunc='sum')
print(pivoted)

transaction_size       Large      Small
region                                 
 East               60487.57    5571.49
 North              85064.74    4280.14
 South              83622.96    3193.92
 West               62340.05    5803.41
East              3219434.87  230642.66
North             3284982.96  209374.20
South             3156244.65  214399.99
West              3184396.75  209894.32
east                98942.97    3405.32
north               62941.31    3634.67
south               83455.78    3962.21
west                70585.27    2875.93


### Q16: How do you reshape the region summary DataFrame from wide format to long format using stack()?

In [17]:
df_size = df.copy()
df_size['transaction_amount'] = pd.to_numeric(df_size['transaction_amount'], errors='coerce')
df_size['transaction_size'] = np.where(df_size['transaction_amount'] > 500, 'Large', 'Small')
pivoted = df_size.pivot_table(index='region', columns='transaction_size', values='transaction_amount', aggfunc='sum')
# Solution:
print(pivoted.stack())

region   transaction_size
 East    Large                 60487.57
         Small                  5571.49
 North   Large                 85064.74
         Small                  4280.14
 South   Large                 83622.96
         Small                  3193.92
 West    Large                 62340.05
         Small                  5803.41
East     Large               3219434.87
         Small                230642.66
North    Large               3284982.96
         Small                209374.20
South    Large               3156244.65
         Small                214399.99
West     Large               3184396.75
         Small                209894.32
east     Large                 98942.97
         Small                  3405.32
north    Large                 62941.31
         Small                  3634.67
south    Large                 83455.78
         Small                  3962.21
west     Large                 70585.27
         Small                  2875.93
dtype: float64

### Q17: How do you perform a non-exact datetime merge (backward) between transaction timestamps and currency rates?

In [18]:
df_txs_time = df.copy()
df_txs_time['transaction_date'] = pd.to_datetime(df_txs_time['transaction_date'], format='mixed')
df_txs_time = df_txs_time.sort_values('transaction_date')
# Create rates table
df_rates = pd.DataFrame({
    'rate_date': pd.to_datetime(['2025-01-01', '2025-06-01', '2026-01-01']),
    'rate': [1.0, 1.05, 1.10]
}).sort_values('rate_date')
# Solution:
print(pd.merge_asof(df_txs_time, df_rates, left_on='transaction_date', right_on='rate_date', direction='backward').head(5))

  transaction_id customer_id merchant_id  ...  amount_z_score  rate_date rate
0       TX104527      C63264       M9586  ...       -1.308953 2025-01-01  1.0
1       TX112396      C23238       M9214  ...        0.977340 2025-01-01  1.0
2       TX105910      C53065       M7797  ...        0.352745 2025-01-01  1.0
3       TX104734      C41358       M6567  ...        1.483010 2025-01-01  1.0
4       TX101254      C28136       M5498  ...       -1.385368 2025-01-01  1.0

[5 rows x 14 columns]


### Q18: How do you generate a cross-join (Cartesian Product) between unique regions and a mock tier table?

In [19]:
df_regions = pd.DataFrame({'region': df['region'].dropna().unique()})
df_tiers = pd.DataFrame({'tier': ['Bronze', 'Silver', 'Gold']})
# Solution:
print(pd.merge(df_regions, df_tiers, how='cross'))

     region    tier
0     North  Bronze
1     North  Silver
2     North    Gold
3      East  Bronze
4      East  Silver
5      East    Gold
6      West  Bronze
7      West  Silver
8      West    Gold
9     South  Bronze
10    South  Silver
11    South    Gold
12    south  Bronze
13    south  Silver
14    south    Gold
15   North   Bronze
16   North   Silver
17   North     Gold
18    East   Bronze
19    East   Silver
20    East     Gold
21    north  Bronze
22    north  Silver
23    north    Gold
24    West   Bronze
25    West   Silver
26    West     Gold
27     west  Bronze
28     west  Silver
29     west    Gold
30   South   Bronze
31   South   Silver
32   South     Gold
33     east  Bronze
34     east  Silver
35     east    Gold


### Q19: How do you track which rows in an outer-joined DataFrame came from which source when joining transactions?

In [20]:
df_l = df.head(3)[['transaction_id', 'customer_id']]
df_r = df.iloc[2:5][['transaction_id', 'transaction_amount']]
# Solution:
print(pd.merge(df_l, df_r, on='transaction_id', how='outer', indicator=True))

  transaction_id customer_id  transaction_amount      _merge
0       TX107002         NaN             1284.68  right_only
1       TX107170      C85674                 NaN   left_only
2       TX108328      C32431              136.66        both
3       TX108563         NaN              124.21  right_only
4       TX110686      C82845                 NaN   left_only


### Q20: Demonstrate the timing speedup of vectorization over .iterrows() when calculating a custom 5% tax fee on transactions.

In [21]:
df_p = df[['transaction_amount']].dropna().head(10000).copy()
t0 = time.time()
res_iter = []
for idx, row in df_p.iterrows():
    res_iter.append(row['transaction_amount'] * 0.05)
t_loop = time.time() - t0
t0 = time.time()
res_vect = df_p['transaction_amount'] * 0.05
t_vect = time.time() - t0
print(f"Iterrows: {t_loop:.5f}s, Vectorized: {t_vect:.5f}s")
print(f"Speedup: {t_loop/max(t_vect, 1e-9):.1f}x")

Iterrows: 0.13408s, Vectorized: 0.00000s
Speedup: 134081840.5x


### Q21: How do you explode a column containing lists into distinct rows?

In [22]:
df_ex = pd.DataFrame({'transaction_id': ['TX100'], 'items': [['Apples', 'Oranges']]})
# Solution:
print(df_ex.explode('items'))

  transaction_id    items
0          TX100   Apples
0          TX100  Oranges


### Q22: How do you extract the numeric user digits from customer_id (e.g. 'C12345' -> '12345') using regex capture groups?

In [23]:
# Solution:
print(df['customer_id'].str.extract(r'C(\d+)').head(5))

       0
0  82845
1  85674
2  32431
3  54057
4  95649


### Q23: How do you load the transactions CSV while treating custom strings like 'Nan' or blank spaces as nulls?

In [24]:
# Solution:
df_na = pd.read_csv('data/raw_transactions.csv', na_values=['Nan', ''])
print("Null counts in amount:", df_na['transaction_amount'].isnull().sum())

Null counts in amount: 738


### Q24: How do you identify duplicate transaction_id rows, showing the duplicate records and dropping them?

In [25]:
# Solution:
duplicates = df[df.duplicated(subset=['transaction_id'], keep=False)]
print(f"Total duplicates found: {len(duplicates)}")
print("Cleaned row count:", len(df.drop_duplicates(subset=['transaction_id'])))

Total duplicates found: 200
Cleaned row count: 14900


### Q25: How do you verify the exact count and percentage of missing values in the transaction_amount column?

In [26]:
# Solution:
missing_cnt = df['transaction_amount'].isna().sum()
missing_pct = (missing_cnt / len(df)) * 100
print(f"Missing Count: {missing_cnt}, Percentage: {missing_pct:.2f}%")

Missing Count: 738, Percentage: 4.92%


### Q26: How do you impute missing transaction_amount values with the overall median transaction amount?

In [27]:
# Solution:
median_amount = df['transaction_amount'].median()
df_imputed = df.copy()
df_imputed['transaction_amount'] = df_imputed['transaction_amount'].fillna(median_amount)
print("Null values remaining:", df_imputed['transaction_amount'].isnull().sum())

Null values remaining: 0


### Q27: What happens when you add Series with index ['a', 'b'] to ['b', 'c'] and how do you fill NaNs with 0?

In [28]:
s1 = pd.Series([10, 20], index=['a', 'b'])
s2 = pd.Series([30, 40], index=['b', 'c'])
# Solution:
print(s1.add(s2, fill_value=0))

a    10.0
b    50.0
c    40.0
dtype: float64


### Q28: How do you select only the columns in our dataset that end with '_id' or start with 'transaction_'?

In [29]:
# Solution:
print(df.filter(regex='.*_id|^transaction_').columns.tolist())

['transaction_id', 'customer_id', 'merchant_id', 'transaction_amount', 'transaction_status', 'transaction_date']


### Q29: How do you extract the top 5 largest transaction amounts in the dataset?

In [30]:
# Solution:
print(df.nlargest(5, 'transaction_amount')[['transaction_id', 'transaction_amount']])

      transaction_id  transaction_amount
12028       TX113019             1999.98
5054        TX101702             1999.85
7337        TX108046             1999.74
11431       TX110350             1999.52
11869       TX111927             1999.36


### Q30: How do you rename the column account_age_months to customer_tenure_months in-place?

In [31]:
df_rn = df.copy()
# Solution:
df_rn.rename(columns={'account_age_months': 'customer_tenure_months'}, inplace=True)
print(df_rn.columns)

Index(['transaction_id', 'customer_id', 'merchant_id', 'transaction_amount',
       'card_type', 'transaction_status', 'device_type',
       'customer_tenure_months', 'transaction_date', 'region', 'is_fraud',
       'amount_z_score'],
      dtype='object')


### Q31: How do you drop the customer_id column and the 1st row of the dataset?

In [32]:
# Solution:
print(df.drop(columns=['customer_id']).drop(0, axis=0).head(2))

  transaction_id merchant_id  ...  is_fraud amount_z_score
1       TX107170       M3868  ...         0      -1.162672
2       TX108328       M1461  ...         0      -1.489005

[2 rows x 11 columns]


### Q32: Show how to retrieve transactions using label-based indexing vs integer position-based indexing.

In [33]:
# Solution:
print("loc label 5:\n", df.loc[5])
print("\niloc position 5:\n", df.iloc[5])

loc label 5:
 transaction_id                   TX113784
customer_id                        C82512
merchant_id                         M8601
transaction_amount                 1263.7
card_type                            Visa
transaction_status              Completed
device_type                        Mobile
account_age_months                     67
transaction_date      2026-04-08 08:43:17
region                              North
is_fraud                                0
amount_z_score                   0.424637
Name: 5, dtype: object

iloc position 5:
 transaction_id                   TX113784
customer_id                        C82512
merchant_id                         M8601
transaction_amount                 1263.7
card_type                            Visa
transaction_status              Completed
device_type                        Mobile
account_age_months                     67
transaction_date      2026-04-08 08:43:17
region                              North
is_fraud            

### Q33: How do you lookup a single transaction's amount at index label 100 using .at and .iat?

In [34]:
# Solution:
print("at label 100:", df.at[100, 'transaction_amount'])
print("iat index 100:", df.iat[100, 2])

at label 100: nan
iat index 100: M9564


### Q34: How do you filter transactions where the amount is > 1000 and the region is 'North' using .query()?

In [35]:
# Solution:
print(df.query('transaction_amount > 1000 and region == "North"').head(3))

   transaction_id customer_id merchant_id  ...  region is_fraud amount_z_score
0        TX110686      C82845       M2697  ...   North        0       0.342910
5        TX113784      C82512       M8601  ...   North        0       0.424637
25       TX105474      C49210       M1126  ...   North        1       1.511748

[3 rows x 12 columns]


### Q35: How do you replace values in transaction_amount where the amount is less than 5 with NaN using .where()?

In [36]:
# Solution:
print(df['transaction_amount'].where(df['transaction_amount'] >= 5).head(5))

0    1216.33
1     324.99
2     136.66
3     124.21
4    1284.68
Name: transaction_amount, dtype: float64


### Q36: How do you create a pivot table showing the count of transactions per region?

In [37]:
# Solution:
print(df.pivot_table(index='region', values='transaction_id', aggfunc='count'))

         transaction_id
region                 
 East                76
 North               82
 South               85
 West                77
East               3665
North              3602
South              3541
West               3565
east                 92
north                65
south                81
west                 69


### Q37: How do you concatenate transactions from 'North' and 'South' regions vertically into one DataFrame?

In [38]:
df_north = df[df.region == 'North']
df_south = df[df.region == 'South']
# Solution:
print("Total concatenated shape:", pd.concat([df_north, df_south], axis=0).shape)

Total concatenated shape: (7143, 12)


### Q38: How do you check unique regions and their frequencies in the dataset?

In [39]:
# Solution:
print("Unique:", df['region'].unique())
print("\nFrequencies:\n", df['region'].value_counts())

Unique: ['North' 'East' 'West' 'South' 'south' ' North ' ' East ' 'north' ' West '
 'west' ' South ' 'east']

Frequencies:
 region
East       3665
North      3602
West       3565
South      3541
east         92
 South       85
 North       82
south        81
 West        77
 East        76
west         69
north        65
Name: count, dtype: int64


### Q39: How do you compute the cumulative sum of transaction amounts ordered by date?

In [40]:
df_sorted = df.copy()
df_sorted['transaction_date'] = pd.to_datetime(df_sorted['transaction_date'], format='mixed')
df_sorted = df_sorted.sort_values('transaction_date')
# Solution:
print(df_sorted['transaction_amount'].cumsum().head(10))

8313      240.57
8702     1798.37
7175     2998.62
11388    4857.55
7038     5054.02
3600     5113.73
854      6323.56
711      7482.11
10943    7642.93
2367     9635.74
Name: transaction_amount, dtype: float64


### Q40: How do you compute the correlation and covariance between account_age_months and transaction_amount?

In [41]:
# Solution:
print("Correlation:\n", df[['account_age_months', 'transaction_amount']].corr())
print("\nCovariance:\n", df[['account_age_months', 'transaction_amount']].cov())

Correlation:
                     account_age_months  transaction_amount
account_age_months            1.000000            0.006784
transaction_amount            0.006784            1.000000

Covariance:
                     account_age_months  transaction_amount
account_age_months         1238.878591          137.793049
transaction_amount          137.793049       331955.476443


### Q41: How do you apply a custom function to label transaction amounts as 'Low', 'Medium', or 'High' dynamically?

In [42]:
def label_amt(val):
    if val < 100: return 'Low'
    elif val < 800: return 'Medium'
    else: return 'High'
# Solution:
print(df['transaction_amount'].apply(label_amt).value_counts())

transaction_amount
High      9313
Medium    5009
Low        678
Name: count, dtype: int64


### Q42: How do you add a column tax_amount (10% of transaction_amount) using .assign()?

In [43]:
# Solution:
df_tax = df.assign(tax_amount=lambda x: x.transaction_amount * 0.1)
print(df_tax[['transaction_amount', 'tax_amount']].head(3))

   transaction_amount  tax_amount
0             1216.33     121.633
1              324.99      32.499
2              136.66      13.666


### Q43: Write code to plot a pie chart of transaction volume per region directly from the DataFrame.

In [44]:
# Solution:
df.groupby('region')['transaction_amount'].sum().plot.pie(autopct='%1.1f%%')
plt.ylabel('')
plt.title("Volume distribution per Region")
plt.show()

<string>:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Q44: Write code to plot a scatter plot of transaction_amount vs account_age_months.

In [45]:
# Solution:
df.plot.scatter(x='account_age_months', y='transaction_amount', alpha=0.5)
plt.title("Amount vs Account Age")
plt.show()

<string>:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Q45: Write code to draw a bar chart of average transaction amounts across different regions.

In [46]:
# Solution:
df.groupby('region')['transaction_amount'].mean().plot.bar()
plt.title("Average Transaction per Region")
plt.show()

<string>:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Q46: Write code to plot a histogram of transaction amounts with 20 bins.

In [47]:
# Solution:
df['transaction_amount'].plot.hist(bins=20)
plt.title("Transaction Amount Histogram")
plt.show()

<string>:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Q47: Write code to draw a box plot of transaction amounts to detect extreme high-value outliers.

In [48]:
# Solution:
df.boxplot(column='transaction_amount')
plt.title("Outliers Detection")
plt.show()

<string>:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Q48: How do you convert the datatype of transaction_id to string and transaction_amount to float32 at once?

In [49]:
# Solution:
df_cast = df.astype({'transaction_id': 'str', 'transaction_amount': 'float32'})
print(df_cast.dtypes)

transaction_id         object
customer_id            object
merchant_id            object
transaction_amount    float32
card_type              object
transaction_status     object
device_type            object
account_age_months      uint8
transaction_date       object
region                 object
is_fraud                int64
amount_z_score        float64
dtype: object


### Q49: How do you merge transactions with a mock customer profile table containing customer signup dates?

In [50]:
df_cust_profile = pd.DataFrame({
    'customer_id': df['customer_id'].dropna().unique(),
    'signup_year': np.random.choice([2022, 2023, 2024], size=len(df['customer_id'].dropna().unique()))
})
# Solution:
merged = pd.merge(df, df_cust_profile, on='customer_id', how='left')
print(merged[['customer_id', 'transaction_amount', 'signup_year']].head(5))

  customer_id  transaction_amount  signup_year
0      C82845             1216.33         2022
1      C85674              324.99         2023
2      C32431              136.66         2022
3      C54057              124.21         2022
4      C95649             1284.68         2022


### Q50: How do you save our cleaned transactions dataset to an Excel file under the sheet name 'Transactions_2026'?

In [51]:
# Solution:
# df.to_excel('transactions_export.xlsx', sheet_name='Transactions_2026', index=False)
print("Mock: df.to_excel('transactions_export.xlsx', sheet_name='Transactions_2026', index=False)")

Mock: df.to_excel('transactions_export.xlsx', sheet_name='Transactions_2026', index=False)
